# Моделирование параметров прибытия подразделений пожарной охраны

## Алгоритмы моделирования

В **Генезис** реализованы следующие алгоритмы:

- `genesis.states.FirstArrivalUnitState`: алгоритм расчета на основе определения времени прибытия от каждой указанной ПСЧ до каждого из узлов ГДС по кратчайшему маршруту (прямая задача)
- `genesis.states.ArrivalTimeMatrixState`: алгоритм расчета на основе заранее рассчитанной матрицы прибытия к каждому из узлов ГДС (обратная задача)

### Интерфейс

Алгоритмы моделирования параметров прибытия подразделений пожарной охраны наследуются от класса `genesis.core.StateBase`.

In [ ]:
class StateBase():
    def __init__(self, state_algorithm, **kwargs):
        ...
    def __call__(self, env, points, area=None, **kwargs):
        ...

### genesis.states.FirstArrivalUnitState

**АРГУМЕНТЫ ИНИЦИАЛИЗАЦИИ**

In [ ]:
FirstArrivalUnitState(
    state_algorithm = MSF,
    weight          = 'travel_time',
    delay           = DELAY_TIME)

* `state_algorithm` -- Функция расчета кратчайших путей от множества источников. По умолчанию передается реализация алгоритма Дейкстры `multi_source_dijkstra` из библиотеки `networkx`, указанная в `genesis.swiss_knife.MSF`
* `weight` -- Алгоритм расчета времени следования ПА. По умолчанию - `"travel_time"`
* `delay` -- Задержка. Например на обслуживание вызова на пожар. По умолчанию указана в `genesis.swiss_knife.DELAY_TIME` и равна 1 мин.

**АРГУМЕНТЫ ВЫЗОВА**

In [ ]:
FirstArrivalUnitState(...)(
    env,
    points,
    area   = None
    )

* `env` (nx.Graph) -- Граф улично-дорожной сети.
* `points` (dict) -- Стартовые узлы. Словарь вида dict(int:str), где ключ - идентификатор узла, значение - его наименование. Может использоваться для указания узлов в которых расположены пожарные подразделения: {1234:'ПСЧ-1'}
* `area` (pd.Series) -- Маска узлов ГДС. Значениями True отмечены узлы ГДС для которых требуется вернуть результат расчета. Если не указана, расчет производится для всех узлов ГДС.
        

**ВОЗВРАЩАЕТ**

times, nearest -> tuple[Series[float], Series[str] | Series]. 
* times - Время прибытия первого подразделения в каждый из узлов ГДС. 
* nearest - Соответствие узлов первому прибывающему подразделению.

### genesis.states.ArrivalTimeMatrixState

**АРГУМЕНТЫ ИНИЦИАЛИЗАЦИИ**

In [ ]:
ArrivalTimeMatrixState(
    matrix,
    state_algorithm = np.min,
    delay           = 0
    )

* `matrix` (pd.DataFrame) -- Матрица времен прибытия, отражающая время прибытия к каждому из объектов из каждого из узлов.
* `state_algorithm` -- Функция оценки времени прибытия из всех возможных стартовых узлов `points`. По умолчанию - `np.min`, что отражает время прибытия первого подразделения.
* `delay` -- Задержка в расчете. Например на обслуживание вызова на пожар. **Предполагается, что уже учтена в `matrix`, поэтому не рекомендуется использовать!**

**ВОЗВРАЩАЕТ**

times, nearest -> tuple[Series[float], Series[str] | Series]. 
* times - Время прибытия первого подразделения в каждый из узлов ГДС. 
* nearest - Соответствие узлов первому прибывающему подразделению.

**АРГУМЕНТЫ ВЫЗОВА**

In [ ]:
ArrivalTimeMatrixState(...)(
    env    = None,
    points = None,
    area   = None
    )

* `env` --  Не используется.
* `points` (dict) -- Стартовые узлы. Словарь вида dict(int:str), где ключ - идентификатор узла, значение - его наименование. Может использоваться для указания узлов в которых расположены пожарные подразделения: {1234:'ПСЧ-1'}
* `area` (pd.Series) -- Маска узлов ГДС. Значениями True отмечены узлы ГДС для которых требуется вернуть результат расчета. Если не указана, расчет производится для всех узлов ГДС.

### genesis.states.get_atm

Функция предназначена для расчета матрицы прибытия, используемой в `ArrivalTimeMatrixState`.

**АРГУМЕНТЫ ВЫЗОВА**

In [ ]:
get_atm(G,
            data: pd.DataFrame    = None,
            data_sample_size: int = None,
            data_node_field: str  = 'node',
            weight: str           = 'travel_time',
            cutoff: float         = None,
            delay: float          = DELAY_TIME,
            target_set: set       = None,
            )

* `G`: nx.MultiDiGraph
    Граф сети городского пожарного хозяйства
* `data`: pd.DataFrame = None
    Данные для расчета. Может быть указан набор данных для которых следует рассчитать,
    например, здания, или набор узлов графа.
    По умолчанию используются все узлы графа
* `data_sample_size`: int = None
    Размер выборки данных для расчета. Количество записей
    data ,которые будут учтены при расчете.
* `data_node_field`: 
    Поле в котором хранится индекс узла, для которого производится расчет.
* `weight`:str или callable  = "travel_time"
            Имя поля содержащего вес ребер, или функция позволяющая вычислять 
            вес динамически.
* `delay`: float, optional = None
    Задержка в расчете. Например на обслуживание вызова на пожар.
    По умолчанию указана в swiss_knife.DELAY_TIME
* `target_set`: set = None
    Целевой сет узлов графа, которые рассматриваются в качестве потенциальных мест размещения.

:::{.callout-note}

#### Прим.

Помните, что аргументы имеющие значение "по умолчанию", нет необходимости повторно указывать явным образом.

Поэтому в большинстве случаев достаточно использовать сокращенный синтаксис, например:

    FirstArrivalUnitState()

вместо:

    FirstArrivalUnitState(state_algorithm = MSF,
                weight = 'travel_time',
                delay = DELAY_TIME,
                **kwargs)

:::

## Модель оценки времени следования

По умолчанию в алгоритмах Генезис используется аргумент `weight = 'travel_time'`. Это означает, что при расчете длины маршрута (в данном случае будет использовано значение  времени следования по каждому из ребер). 

Пользователь может указать иное значение соответствующее названию атрибута данных ребер ГДС. Например, при указании `weight = 'length'`, расчет будет выполнен для кратчайшего расстояния в метрах (длина ребер ГДС).

Это упрощенная модель не учитывающая множества дополнительных факторов оказывающих влияние на время прибытия.

Однако пользователь может использовать и более сложные модели оценки. Так, например, в качестве аргумента `weight` может быть передана функция с интерфейсом:

In [ ]:
def func(u, v, d):
    # Получаем вес ребра по `travel_time`
    edge_wt = d.get("travel_time", 1)

    # Проверяем не является ли целевой узел узлом на котором имеется светофор
    node_type = G.nodes[v].get("highway", None)
    # Если светофор:
    if node_type == 'traffic_signals':
        # То добавляем к времени следования штраф за задержку 0.1 мин
        return edge_wt + 0.1
    
    # Если нет, то возвращаем только время следования по ребру ГДС
    return edge_wt

Данная функция позволяет модифицировать модель расчета времени следования подразделений пожарной охраны за счет внедрения учета штрафа за преодоление участков уличной сети с наличием светофоров.

Более подробно о `weight` можно прочесть в [документации networkx](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.dijkstra_path.html#networkx.algorithms.shortest_paths.weighted.dijkstra_path)

## Алгоритмы поиска кратчайшего пути

По умолчанию в **Генезис** используется алгоритм Дейкстры в его реализации для расчета вех маршрутов от нескольких стартовых точек [nx.multi_source_dijkstra](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.multi_source_dijkstra.html).

Данный алгоритм имеет интерфейс:

In [ ]:
func(G: Graph, sources: Any, target: Any | None = None, cutoff: Any | None = None, 
                                    weight: str = "weight") -> (dict, dict)

При необходимости пользователь может:

1. Доработать существующий алгоритм воспользовавшись исходным кодом библиотеки `networkx` (при условии соблюдения правил использования прописанных в лицензионных соглашениях)

2. Написать собственный соблюдая интерфейс функции.